# 4. Attitude representation: Euler angles vs SO(3)

`simulation/dynamics.py` carries two attitude integrators: the Euler-angle path
(`_rigid_body_derivatives`, ported verbatim from `baseline_code.py`) and a
quaternion/SO(3) path (`_rigid_body_derivatives_so3`, added for the
attitude-control work). A torque-free tumble (`B = 0`, dipole moment 0, so
angular momentum is conserved) runs through both to check:

1. the Euler-angle path degrades as the excursion grows: its ZYX chart is
   singular at pitch `= +-90 deg`, and the integrator drops the `-omega x I*omega`
   precession term;
2. near level, the two paths agree to sub-milliradian, so keeping the
   Euler-angle path for levitation work costs no attitude error.

The section 1 divergence trajectory reaches about -100 deg pitch, past the
Euler-rate singularity, so large-excursion runs must use SO(3). Machinery in
`nblib/attitude_representation.py`.

In [ ]:
from nblib.bootstrap import REPO_ROOT
from nblib import attitude_representation as ar

import numpy as np
import matplotlib.pyplot as plt

from geometry.transforms import rotation_matrix_from_euler
from geometry.quaternion import quat_to_matrix

FIG_DIR = REPO_ROOT / "docs" / "results_figures"

# A pod with three distinct principal moments, so a tumble precesses.
POD_NAME = "Three Stacks Front-Mid-Back"
pod, I_body, I_body_inv, mass = ar.pod_body_inertia(POD_NAME)
print(f"pod = {POD_NAME!r}")
print(f"I_body principal moments (kg m^2) = {np.round(np.diag(I_body), 9)}")
print(f"total mass (kg) = {mass:.5f}")

## 4a. The tumble

`omega0 = (0.4, 6.0, 0.1) rad/s`, a spin about the intermediate principal axis,
so the motion precesses strongly and carries the extracted pitch into the
`+-90 deg` gimbal band. Solver tolerance is tight (`rtol = 1e-10`), so any error
is the method, not the step size.

Top: ZYX angles recovered from the SO(3) attitude; the `+-180 deg` roll/yaw jumps
near `|pitch| = 90 deg` are the chart singularity, not a physical event. Bottom:
`|L_world(t) - L_world(0)| / |L0|`. SO(3) holds angular momentum to ~1e-13, the
Euler-angle path climbs past 1.

In [ ]:
OMEGA0_TUMBLE = np.array([0.4, 6.0, 0.1])
T_TUMBLE = 6.0

sol_e = ar.run_tumble(I_body, I_body_inv, mass, OMEGA0_TUMBLE,
                      (0.0, T_TUMBLE), "euler", rtol=1e-10, atol=1e-13)
sol_s = ar.run_tumble(I_body, I_body_inv, mass, OMEGA0_TUMBLE,
                      (0.0, T_TUMBLE), "so3", rtol=1e-10, atol=1e-13)

ts = np.linspace(0.0, T_TUMBLE, 1200)
ye = sol_e.sol(ts)
ys = sol_s.sol(ts)

L0 = I_body @ OMEGA0_TUMBLE

rot_e = [
    rotation_matrix_from_euler(ye[6, i], ye[7, i], ye[8, i])
    for i in range(ts.size)
]
rot_s = [
    quat_to_matrix(ys[6:10, i])
    for i in range(ts.size)
]

L_e = ar.angular_momentum(rot_e, ye[9:12, :], I_body)
L_s = ar.angular_momentum(rot_s, ys[10:13, :], I_body)

err_e = ar.momentum_drift(L_e, L0)
err_s = ar.momentum_drift(L_s, L0)

ang_s = ar.euler_angles(rot_s)

near_90 = np.degrees(np.abs(ang_s[1])).max()
print(f"extracted |pitch| reaches {near_90:.1f} deg  (gimbal band = 90 deg)")
print(f"Euler-angle path max |dL|/|L0| = {err_e.max():.3f}")
print(f"SO(3)       path max |dL|/|L0| = {err_s.max():.2e}")
CHECK1 = err_s.max() < 1e-9 < err_e.max()
print(f"CHECK 1: SO(3) conserves L to <1e-9 while the Euler-angle path does not "
      f"-> {'PASS' if CHECK1 else 'FAIL'}")

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8.2, 6.4), sharex=True)

# Top: ZYX angles recovered from the SO(3) attitude.
ax0.axhline(90, color="grey", lw=0.8, ls=":")
ax0.axhline(-90, color="grey", lw=0.8, ls=":")
ax0.plot(ts, np.degrees(ang_s[0]), label="roll", color="blue", lw=1.3)
ax0.plot(ts, np.degrees(ang_s[1]), label="pitch", color="green", lw=1.6)
ax0.plot(ts, np.degrees(ang_s[2]), label="yaw", color="red", lw=1.0, alpha=0.8)
ax0.set_ylabel("ZYX angle (deg)")
_omega_str = "(" + ", ".join(f"{v:g}" for v in OMEGA0_TUMBLE) + ")"
ax0.set_title(f"Torque-free tumble, omega0 = {_omega_str} rad/s")
ax0.legend(fontsize=9, ncol=3, loc="lower right")
ax0.text(0.02, 0.92, "roll / yaw jump +-180 deg as |pitch| -> 90 deg\n(ZYX chart singular there)",
         transform=ax0.transAxes, fontsize=8.5, va="top", color="grey")

# Bottom: angular-momentum drift, log scale.
ax1.semilogy(ts, np.clip(err_e, 1e-16, None), label="Euler-angle path",
             color="red", lw=1.6)
ax1.semilogy(ts, np.clip(err_s, 1e-16, None), label="SO(3) path",
             color="blue", lw=1.6)
ax1.set_xlabel("time (s)")
ax1.set_ylabel(r"$|L_{world}(t) - L_{world}(0)| / |L_0|$")
ax1.legend(fontsize=9, loc="center right")
ax1.set_ylim(1e-16, 1e1)

for ax in (ax0, ax1):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "attitude_repr_gimbal_lock.png", dpi=150)
plt.show()

Recovered ZYX angles (top) show the chart singularity as the body swings into
the `+-90 deg` band; the quaternion state passes through smoothly. Angular
momentum (bottom, log scale) is the frame-independent check: SO(3) holds it to
~1e-13, the Euler-angle path loses it entirely.

## 4b. Near level, no penalty

Sweeps the total rotation angle of a light tumble (`omega0` proportional to
`(0.15, 1.0, 0.05)`) from 2 to 80 deg, recording the largest angle-wise gap
between the two integrators along each trajectory. Below ~10 deg they agree to
well under a milliradian.

In [ ]:
base_dir = np.array([0.15, 1.0, 0.05])
base_dir = base_dir / np.linalg.norm(base_dir)
SPIN_RATE = 6.0

excursions_deg = np.array([2, 5, 10, 15, 20, 30, 40, 60, 80], dtype=float)
max_gap_rad = np.zeros_like(excursions_deg)

for k, total_deg in enumerate(excursions_deg):
    T = np.radians(total_deg) / SPIN_RATE
    omega0 = base_dir * SPIN_RATE

    se = ar.run_tumble(I_body, I_body_inv, mass, omega0, (0.0, T), "euler",
                       rtol=1e-11, atol=1e-13, max_step=T / 80)
    ss = ar.run_tumble(I_body, I_body_inv, mass, omega0, (0.0, T), "so3",
                       rtol=1e-11, atol=1e-13, max_step=T / 80)

    tk = np.linspace(0.0, T, 200)
    ek = se.sol(tk)
    sk = ss.sol(tk)

    ang_e = ek[6:9, :]
    ang_s = ar.euler_angles([
        quat_to_matrix(sk[6:10, i])
        for i in range(tk.size)
    ])
    max_gap_rad[k] = np.max(np.abs(ang_e - ang_s))

for d, g in zip(excursions_deg, max_gap_rad):
    print(f"  excursion {d:5.0f} deg -> max |Euler-angle - SO(3)| = {g:.2e} rad")

NEAR_LEVEL = max_gap_rad[excursions_deg <= 10].max()
CHECK2 = NEAR_LEVEL < 1e-3
print(f"CHECK 2: near-level (<=10 deg) agreement {NEAR_LEVEL:.2e} rad < 1e-3 "
      f"-> {'PASS' if CHECK2 else 'FAIL'}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.4))

ax.loglog(excursions_deg, np.clip(max_gap_rad, 1e-16, None), "o-",
          color="blue", lw=1.6, ms=5)
ax.axvspan(excursions_deg.min(), 10, color="green", alpha=0.10)
ax.text(3.0, max_gap_rad.max() * 0.3, "near-level regime\n(levitation, section 1)",
        fontsize=9, color="green")
ax.axhline(1e-3, color="grey", lw=0.8, ls=":")
ax.text(excursions_deg.max(), 1.2e-3, "1 mrad", ha="right", va="bottom",
        fontsize=8.5, color="grey")

ax.set_xlabel("rotation excursion (deg)")
ax.set_ylabel("max |Euler-angle - SO(3)| (rad)")
ax.set_title("Euler-angle vs SO(3): max angle gap")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "attitude_repr_agreement.png", dpi=150)
plt.show()

Largest angle-wise gap between the two integrators vs rotation excursion. It
grows roughly quadratically; in the near-level band (shaded) it stays below a
milliradian, so the Euler-angle path introduces no attitude error for levitation
work. Only large-excursion runs need SO(3).

## Summary

Check 1: SO(3) conserves `L` through the gimbal band, the Euler-angle path does
not. Check 2: near-level (`<= 10 deg`) agreement is below 1 mrad.

The Euler-angle path is kept for near-level levitation and section 1 work. Any
large-excursion trajectory (the section 1 divergence, an attitude-control
transient, a tumble) uses `_rigid_body_derivatives_so3`.